# Week 09 — Decision Trees: C4.5 and CART

From-scratch implementations (no scikit-learn) plus a scikit-learn comparison.

**Questions covered**
1. C4.5 from scratch on the Play Tennis table, then classify a new sample.
2. CART from scratch on the same table, then classify a new sample.
3. C4.5 and CART, with and without scikit-learn, on the loan-approval table.
4. Additional: sklearn entropy (C4.5-like) and sklearn gini (CART) on Play Tennis, compared with the from-scratch trees.


## 1. Data

In [1]:
import math
from collections import Counter, defaultdict

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

TENNIS = [
    {"Outlook": "Sunny",    "Temp": 85, "Humidity": 85, "Wind": "Weak",   "Decision": "No"},
    {"Outlook": "Sunny",    "Temp": 80, "Humidity": 90, "Wind": "Strong", "Decision": "No"},
    {"Outlook": "Overcast", "Temp": 83, "Humidity": 78, "Wind": "Weak",   "Decision": "Yes"},
    {"Outlook": "Rain",     "Temp": 70, "Humidity": 96, "Wind": "Weak",   "Decision": "Yes"},
    {"Outlook": "Rain",     "Temp": 68, "Humidity": 80, "Wind": "Weak",   "Decision": "Yes"},
    {"Outlook": "Rain",     "Temp": 65, "Humidity": 70, "Wind": "Strong", "Decision": "No"},
    {"Outlook": "Overcast", "Temp": 64, "Humidity": 65, "Wind": "Strong", "Decision": "Yes"},
    {"Outlook": "Sunny",    "Temp": 72, "Humidity": 95, "Wind": "Weak",   "Decision": "No"},
    {"Outlook": "Sunny",    "Temp": 69, "Humidity": 70, "Wind": "Weak",   "Decision": "Yes"},
    {"Outlook": "Rain",     "Temp": 75, "Humidity": 80, "Wind": "Weak",   "Decision": "Yes"},
    {"Outlook": "Sunny",    "Temp": 75, "Humidity": 70, "Wind": "Strong", "Decision": "Yes"},
    {"Outlook": "Overcast", "Temp": 72, "Humidity": 90, "Wind": "Strong", "Decision": "Yes"},
    {"Outlook": "Overcast", "Temp": 81, "Humidity": 75, "Wind": "Weak",   "Decision": "Yes"},
    {"Outlook": "Rain",     "Temp": 71, "Humidity": 80, "Wind": "Strong", "Decision": "No"},
]
TENNIS_FEATURES = ["Outlook", "Temp", "Humidity", "Wind"]
TENNIS_CAT = {"Outlook", "Wind"}
TARGET = "Decision"

NEW_TENNIS = [
    {"Outlook": "Sunny",    "Temp": 70, "Humidity": 80, "Wind": "Weak"},
    {"Outlook": "Rain",     "Temp": 68, "Humidity": 75, "Wind": "Strong"},
    {"Outlook": "Overcast", "Temp": 75, "Humidity": 90, "Wind": "Strong"},
]

LOAN = [
    {"Income": "Low",    "Credit": "Good", "Loan": "Yes"},
    {"Income": "Low",    "Credit": "Bad",  "Loan": "No"},
    {"Income": "Medium", "Credit": "Good", "Loan": "Yes"},
    {"Income": "Medium", "Credit": "Bad",  "Loan": "Yes"},
    {"Income": "High",   "Credit": "Good", "Loan": "Yes"},
    {"Income": "High",   "Credit": "Bad",  "Loan": "No"},
]
LOAN_FEATURES = ["Income", "Credit"]
LOAN_TARGET = "Loan"
NEW_LOAN = [
    {"Income": "Medium", "Credit": "Bad"},
    {"Income": "High",   "Credit": "Bad"},
    {"Income": "Low",    "Credit": "Good"},
]

display(pd.DataFrame(TENNIS))


,Outlook,Temp,Humidity,Wind,Decision
0,Sunny,85,85,Weak,No
1,Sunny,80,90,Strong,No
2,Overcast,83,78,Weak,Yes
3,Rain,70,96,Weak,Yes
4,Rain,68,80,Weak,Yes
5,Rain,65,70,Strong,No
6,Overcast,64,65,Strong,Yes
7,Sunny,72,95,Weak,No
8,Sunny,69,70,Weak,Yes
9,Rain,75,80,Weak,Yes


## 2. Impurity measures

In [2]:
def majority(rows, target):
    return Counter(r[target] for r in rows).most_common(1)[0][0]


def entropy(rows, target):
    n = len(rows)
    if n == 0:
        return 0.0
    counts = Counter(r[target] for r in rows)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())


def gini(rows, target):
    n = len(rows)
    if n == 0:
        return 0.0
    counts = Counter(r[target] for r in rows)
    return 1.0 - sum((c / n) ** 2 for c in counts.values())


def split_info(parts):
    total = sum(len(p) for p in parts)
    return -sum((len(p) / total) * math.log2(len(p) / total) for p in parts if p)


def train_accuracy(rows, predict, target):
    correct = sum(predict(r) == r[target] for r in rows)
    return correct, len(rows)


print(f"Play Tennis  entropy = {entropy(TENNIS, TARGET):.4f}   gini = {gini(TENNIS, TARGET):.4f}")
print(f"Loan         entropy = {entropy(LOAN, LOAN_TARGET):.4f}   gini = {gini(LOAN, LOAN_TARGET):.4f}")


Play Tennis  entropy = 0.9403   gini = 0.4592
Loan         entropy = 0.9183   gini = 0.4444


## 3. C4.5 from scratch

`c45_choose` builds every legal split, drops attributes whose information gain is below the average, then picks the highest gain ratio.


In [3]:
def candidate_splits(rows, features, cat_features, target):
    """Legal splits scored by information gain. Numeric: IG-maximising threshold only."""
    base = entropy(rows, target)
    n = len(rows)
    cands = []
    for f in features:
        if f in cat_features:
            groups = defaultdict(list)
            for r in rows:
                groups[r[f]].append(r)
            parts = list(groups.values())
            if len(parts) < 2:
                continue
            gain = base - sum((len(p) / n) * entropy(p, target) for p in parts)
            si = split_info(parts)
            if gain <= 1e-12 or si == 0:
                continue
            cands.append({
                "feature": f, "gain": gain, "split_info": si, "gain_ratio": gain / si,
                "spec": ("cat", None), "groups": dict(groups),
            })
        else:
            values = sorted(set(r[f] for r in rows))
            best_thr = None
            for i in range(len(values) - 1):
                thr = (values[i] + values[i + 1]) / 2.0
                left = [r for r in rows if r[f] <= thr]
                right = [r for r in rows if r[f] > thr]
                if not left or not right:
                    continue
                gain = base - (len(left) / n) * entropy(left, target) - (len(right) / n) * entropy(right, target)
                if best_thr is None or gain > best_thr["gain"]:
                    best_thr = {
                        "feature": f, "gain": gain,
                        "split_info": split_info([left, right]),
                        "spec": ("num", thr),
                        "groups": {"<=": left, ">": right},
                    }
            if best_thr and best_thr["gain"] > 1e-12 and best_thr["split_info"] > 0:
                best_thr["gain_ratio"] = best_thr["gain"] / best_thr["split_info"]
                cands.append(best_thr)
    return cands


def c45_choose(rows, features, cat_features, target):
    cands = candidate_splits(rows, features, cat_features, target)
    if not cands:
        return None
    avg_gain = sum(c["gain"] for c in cands) / len(cands)
    eligible = [c for c in cands if c["gain"] >= avg_gain - 1e-12] or cands
    return max(eligible, key=lambda c: c["gain_ratio"])


def build_c45(rows, features, cat_features, target, depth=0, max_depth=8):
    labels = [r[target] for r in rows]
    dist = dict(Counter(labels))
    if len(set(labels)) == 1 or not features or depth >= max_depth or len(rows) < 2:
        return {"leaf": True, "label": majority(rows, target), "n": len(rows), "dist": dist}
    chosen = c45_choose(rows, features, cat_features, target)
    if chosen is None:
        return {"leaf": True, "label": majority(rows, target), "n": len(rows), "dist": dist}
    next_features = [f for f in features if f != chosen["feature"] or f not in cat_features]
    node = {
        "leaf": False,
        "feature": chosen["feature"],
        "spec": chosen["spec"],
        "gain": round(chosen["gain"], 4),
        "gain_ratio": round(chosen["gain_ratio"], 4),
        "n": len(rows),
        "children": {},
    }
    for key, subset in chosen["groups"].items():
        node["children"][key] = build_c45(subset, next_features, cat_features, target, depth + 1, max_depth)
    return node


def predict_c45(tree, sample):
    node = tree
    while not node["leaf"]:
        feat = node["feature"]
        kind, thr = node["spec"]
        if kind == "cat":
            key = sample[feat]
            if key not in node["children"]:
                key = max(node["children"], key=lambda k: node["children"][k]["n"])
        else:
            key = "<=" if sample[feat] <= thr else ">"
        node = node["children"][key]
    return node["label"]


def format_c45(node, indent=0):
    pad = "  " * indent
    if node["leaf"]:
        return f"{pad}Leaf -> {node['label']}   n={node['n']}  {node['dist']}"
    kind, thr = node["spec"]
    lines = [f"{pad}[{node['feature']}]  IG={node['gain']}  gainRatio={node['gain_ratio']}  n={node['n']}"]
    for key, child in node["children"].items():
        if kind == "num":
            label = f"{node['feature']} <= {thr:g}" if key == "<=" else f"{node['feature']} > {thr:g}"
        else:
            label = f"{node['feature']} = {key}"
        lines.append(f"{pad}  -> {label}")
        lines.append(format_c45(child, indent + 2))
    return "\n".join(lines)


### Q1 — build the C4.5 tree and classify new samples

In [4]:
cands = candidate_splits(TENNIS, TENNIS_FEATURES, TENNIS_CAT, TARGET)
avg = sum(c["gain"] for c in cands) / len(cands)
print(f"Entropy(S) = {entropy(TENNIS, TARGET):.4f}")
print(f"Average IG = {avg:.4f}  (only attributes with IG >= average are eligible)\n")
for c in sorted(cands, key=lambda x: -x["gain_ratio"]):
    extra = f" thr={c['spec'][1]:g}" if c["spec"][0] == "num" else ""
    flag = "ELIGIBLE" if c["gain"] >= avg - 1e-12 else "filtered out"
    print(f"  {c['feature']:10s}{extra:12s}  IG={c['gain']:.4f}  splitInfo={c['split_info']:.4f}  "
          f"gainRatio={c['gain_ratio']:.4f}  {flag}")

tree_c45 = build_c45(TENNIS, TENNIS_FEATURES, TENNIS_CAT, TARGET)
print("\nLearned C4.5 tree:")
print(format_c45(tree_c45))
correct, total = train_accuracy(TENNIS, lambda s: predict_c45(tree_c45, s), TARGET)
print(f"\nTraining accuracy: {correct}/{total}")
print("\nNew samples:")
for s in NEW_TENNIS:
    print(f"  {s}  ->  {predict_c45(tree_c45, s)}")


Entropy(S) = 0.9403
Average IG = 0.1276  (only attributes with IG >= average are eligible)

  Temp       thr=84       IG=0.1134  splitInfo=0.3712  gainRatio=0.3055  filtered out
  Outlook                 IG=0.2467  splitInfo=1.5774  gainRatio=0.1564  ELIGIBLE
  Humidity   thr=82.5     IG=0.1022  splitInfo=0.9403  gainRatio=0.1087  filtered out
  Wind                    IG=0.0481  splitInfo=0.9852  gainRatio=0.0488  filtered out

Learned C4.5 tree:
[Outlook]  IG=0.2467  gainRatio=0.1564  n=14
  -> Outlook = Sunny
    [Humidity]  IG=0.971  gainRatio=1.0  n=5
      -> Humidity <= 77.5
        Leaf -> Yes   n=2  {'Yes': 2}
      -> Humidity > 77.5
        Leaf -> No   n=3  {'No': 3}
  -> Outlook = Overcast
    Leaf -> Yes   n=4  {'Yes': 4}
  -> Outlook = Rain
    [Wind]  IG=0.971  gainRatio=1.0  n=5
      -> Wind = Weak
        Leaf -> Yes   n=3  {'Yes': 3}
      -> Wind = Strong
        Leaf -> No   n=2  {'No': 2}

Training accuracy: 14/14

New samples:
  {'Outlook': 'Sunny', 'Temp': 70, 

## 4. CART from scratch

Every categorical attribute is tried as a binary grouping of its values. Every numeric attribute is tried at the midpoint between sorted unique values. The split with the smallest weighted Gini is kept, and the same feature may be reused.


In [5]:
def cart_best_split(rows, features, cat_features, target):
    n = len(rows)
    best = None
    for f in features:
        if f in cat_features:
            values = sorted(set(r[f] for r in rows))
            if len(values) < 2:
                continue
            seen = set()
            m = len(values)
            for mask in range(1, (1 << m) - 1):
                left_vals = frozenset(values[i] for i in range(m) if mask & (1 << i))
                canon = tuple(sorted(left_vals))
                comp = tuple(sorted(set(values) - set(left_vals)))
                key = min(canon, comp)
                if key in seen:
                    continue
                seen.add(key)
                left = [r for r in rows if r[f] in left_vals]
                right = [r for r in rows if r[f] not in left_vals]
                if not left or not right:
                    continue
                cost = (len(left) / n) * gini(left, target) + (len(right) / n) * gini(right, target)
                if best is None or cost < best[0] - 1e-15:
                    best = (cost, f, ("cat", set(left_vals)), left, right)
        else:
            values = sorted(set(r[f] for r in rows))
            for i in range(len(values) - 1):
                thr = (values[i] + values[i + 1]) / 2.0
                left = [r for r in rows if r[f] <= thr]
                right = [r for r in rows if r[f] > thr]
                if not left or not right:
                    continue
                cost = (len(left) / n) * gini(left, target) + (len(right) / n) * gini(right, target)
                if best is None or cost < best[0] - 1e-15:
                    best = (cost, f, ("num", thr), left, right)
    return best


def build_cart(rows, features, cat_features, target, depth=0, max_depth=8, min_samples_split=2):
    labels = [r[target] for r in rows]
    dist = dict(Counter(labels))
    if len(set(labels)) == 1 or not features or depth >= max_depth or len(rows) < min_samples_split:
        return {"leaf": True, "label": majority(rows, target), "n": len(rows), "dist": dist}
    best = cart_best_split(rows, features, cat_features, target)
    if best is None or best[0] >= gini(rows, target) - 1e-12:
        return {"leaf": True, "label": majority(rows, target), "n": len(rows), "dist": dist}
    cost, feat, spec, left, right = best
    return {
        "leaf": False,
        "feature": feat,
        "spec": spec,
        "weighted_gini": round(cost, 4),
        "n": len(rows),
        "children": {
            "left": build_cart(left, features, cat_features, target, depth + 1, max_depth, min_samples_split),
            "right": build_cart(right, features, cat_features, target, depth + 1, max_depth, min_samples_split),
        },
    }


def predict_cart(tree, sample):
    node = tree
    while not node["leaf"]:
        kind, payload = node["spec"]
        if kind == "num":
            go_left = sample[node["feature"]] <= payload
        else:
            go_left = sample[node["feature"]] in payload
        node = node["children"]["left" if go_left else "right"]
    return node["label"]


def format_cart(node, indent=0):
    pad = "  " * indent
    if node["leaf"]:
        return f"{pad}Leaf -> {node['label']}   n={node['n']}  {node['dist']}"
    kind, payload = node["spec"]
    if kind == "num":
        left_lbl = f"{node['feature']} <= {payload:g}"
        right_lbl = f"{node['feature']} > {payload:g}"
    else:
        left_lbl = f"{node['feature']} in {sorted(payload)}"
        right_lbl = f"{node['feature']} not in {sorted(payload)}"
    lines = [f"{pad}[{node['feature']}]  weightedGini={node['weighted_gini']}  n={node['n']}"]
    lines.append(f"{pad}  -> {left_lbl}")
    lines.append(format_cart(node["children"]["left"], indent + 2))
    lines.append(f"{pad}  -> {right_lbl}")
    lines.append(format_cart(node["children"]["right"], indent + 2))
    return "\n".join(lines)


### Q2 — build the CART tree and classify new samples

In [6]:
print(f"Gini(S) = {gini(TENNIS, TARGET):.4f}")
tree_cart = build_cart(TENNIS, TENNIS_FEATURES, TENNIS_CAT, TARGET)
print("\nLearned CART tree:")
print(format_cart(tree_cart))
correct, total = train_accuracy(TENNIS, lambda s: predict_cart(tree_cart, s), TARGET)
print(f"\nTraining accuracy: {correct}/{total}")
print("\nNew samples:")
for s in NEW_TENNIS:
    print(f"  {s}  ->  {predict_cart(tree_cart, s)}")


Gini(S) = 0.4592

Learned CART tree:
[Outlook]  weightedGini=0.3571  n=14
  -> Outlook in ['Overcast']
    Leaf -> Yes   n=4  {'Yes': 4}
  -> Outlook not in ['Overcast']
    [Temp]  weightedGini=0.375  n=10
      -> Temp <= 77.5
        [Temp]  weightedGini=0.3571  n=8
          -> Temp <= 66.5
            Leaf -> No   n=1  {'No': 1}
          -> Temp > 66.5
            [Temp]  weightedGini=0.2857  n=7
              -> Temp <= 70.5
                Leaf -> Yes   n=3  {'Yes': 3}
              -> Temp > 70.5
                [Temp]  weightedGini=0.0  n=4
                  -> Temp <= 73.5
                    Leaf -> No   n=2  {'No': 2}
                  -> Temp > 73.5
                    Leaf -> Yes   n=2  {'Yes': 2}
      -> Temp > 77.5
        Leaf -> No   n=2  {'No': 2}

Training accuracy: 14/14

New samples:
  {'Outlook': 'Sunny', 'Temp': 70, 'Humidity': 80, 'Wind': 'Weak'}  ->  Yes
  {'Outlook': 'Rain', 'Temp': 68, 'Humidity': 75, 'Wind': 'Strong'}  ->  Yes
  {'Outlook': 'Overcast', 'T

## 5. scikit-learn comparison on Play Tennis

Categorical columns are one-hot encoded so Outlook is not treated as an ordered number. `criterion="entropy"` is binary information gain; `criterion="gini"` is CART.


In [7]:
def sklearn_tree(rows, cat_cols, num_cols, target, criterion):
    df = pd.DataFrame(rows)
    enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_cat = enc.fit_transform(df[cat_cols])
    cat_names = list(enc.get_feature_names_out(cat_cols))
    X = pd.DataFrame(X_cat, columns=cat_names)
    for col in num_cols:
        X[col] = df[col].values
    clf = DecisionTreeClassifier(criterion=criterion, random_state=0)
    clf.fit(X, df[target])
    return clf, enc, list(X.columns), cat_cols, num_cols


def sklearn_predict(bundle, sample):
    clf, enc, feature_names, cat_cols, num_cols = bundle
    row = pd.DataFrame([{c: sample[c] for c in cat_cols}])
    X_cat = enc.transform(row)
    data = dict(zip(enc.get_feature_names_out(cat_cols), X_cat[0]))
    for col in num_cols:
        data[col] = sample[col]
    X = pd.DataFrame([{name: data.get(name, 0) for name in feature_names}])
    return clf.predict(X)[0]


sk_ent = sklearn_tree(TENNIS, ["Outlook", "Wind"], ["Temp", "Humidity"], TARGET, "entropy")
sk_gini = sklearn_tree(TENNIS, ["Outlook", "Wind"], ["Temp", "Humidity"], TARGET, "gini")

print("sklearn criterion='entropy' (binary IG, not gain ratio):")
print(export_text(sk_ent[0], feature_names=sk_ent[2]))
print("sklearn criterion='gini' (CART):")
print(export_text(sk_gini[0], feature_names=sk_gini[2]))

rows = []
for s in NEW_TENNIS:
    rows.append({
        "Outlook": s["Outlook"], "Temp": s["Temp"], "Humidity": s["Humidity"], "Wind": s["Wind"],
        "C4.5": predict_c45(tree_c45, s),
        "CART": predict_cart(tree_cart, s),
        "sklearn entropy": sklearn_predict(sk_ent, s),
        "sklearn gini": sklearn_predict(sk_gini, s),
    })
display(pd.DataFrame(rows))
X_ent = pd.DataFrame(
    __import__("numpy").column_stack([
        sk_ent[1].transform(pd.DataFrame(TENNIS)[["Outlook", "Wind"]]),
        pd.DataFrame(TENNIS)[["Temp", "Humidity"]].to_numpy(),
    ]),
    columns=sk_ent[2],
)
X_gini = pd.DataFrame(
    __import__("numpy").column_stack([
        sk_gini[1].transform(pd.DataFrame(TENNIS)[["Outlook", "Wind"]]),
        pd.DataFrame(TENNIS)[["Temp", "Humidity"]].to_numpy(),
    ]),
    columns=sk_gini[2],
)
y = pd.DataFrame(TENNIS)[TARGET]
print("sklearn training accuracy  entropy =", round(sk_ent[0].score(X_ent, y), 4),
      " gini =", round(sk_gini[0].score(X_gini, y), 4))


sklearn criterion='entropy' (binary IG, not gain ratio):
|--- Outlook_Overcast <= 0.50
|   |--- Temp <= 77.50
|   |   |--- Temp <= 73.50
|   |   |   |--- Wind_Strong <= 0.50
|   |   |   |   |--- Temp <= 71.00
|   |   |   |   |   |--- class: Yes
|   |   |   |   |--- Temp >  71.00
|   |   |   |   |   |--- class: No
|   |   |   |--- Wind_Strong >  0.50
|   |   |   |   |--- class: No
|   |   |--- Temp >  73.50
|   |   |   |--- class: Yes
|   |--- Temp >  77.50
|   |   |--- class: No
|--- Outlook_Overcast >  0.50
|   |--- class: Yes

sklearn criterion='gini' (CART):
|--- Outlook_Overcast <= 0.50
|   |--- Temp <= 77.50
|   |   |--- Temp <= 66.50
|   |   |   |--- class: No
|   |   |--- Temp >  66.50
|   |   |   |--- Temp <= 70.50
|   |   |   |   |--- class: Yes
|   |   |   |--- Temp >  70.50
|   |   |   |   |--- Temp <= 73.50
|   |   |   |   |   |--- class: No
|   |   |   |   |--- Temp >  73.50
|   |   |   |   |   |--- class: Yes
|   |--- Temp >  77.50
|   |   |--- class: No
|--- Outlook_Over

,Outlook,Temp,Humidity,Wind,C4.5,CART,sklearn entropy,sklearn gini
0,Sunny,70,80,Weak,No,Yes,Yes,Yes
1,Rain,68,75,Strong,No,Yes,No,Yes
2,Overcast,75,90,Strong,Yes,Yes,Yes,Yes


sklearn training accuracy  entropy = 1.0  gini = 1.0


## 6. Q3 — loan data, with and without scikit-learn

Attributes are Income (Low, Medium, High) and Credit (Good, Bad). Target is Loan Approved.


In [8]:
display(pd.DataFrame(LOAN))

loan_c45 = build_c45(LOAN, LOAN_FEATURES, set(LOAN_FEATURES), LOAN_TARGET)
loan_cart = build_cart(LOAN, LOAN_FEATURES, set(LOAN_FEATURES), LOAN_TARGET)
print("From-scratch C4.5:")
print(format_c45(loan_c45))
print("\nFrom-scratch CART:")
print(format_cart(loan_cart))

sk_le = sklearn_tree(LOAN, LOAN_FEATURES, [], LOAN_TARGET, "entropy")
sk_lg = sklearn_tree(LOAN, LOAN_FEATURES, [], LOAN_TARGET, "gini")
print("\nsklearn entropy:")
print(export_text(sk_le[0], feature_names=sk_le[2]))
print("sklearn gini:")
print(export_text(sk_lg[0], feature_names=sk_lg[2]))

c1, n1 = train_accuracy(LOAN, lambda s: predict_c45(loan_c45, s), LOAN_TARGET)
c2, n2 = train_accuracy(LOAN, lambda s: predict_cart(loan_cart, s), LOAN_TARGET)
print(f"Training accuracy  C4.5 {c1}/{n1}   CART {c2}/{n2}   "
      f"sklearn entropy {sk_le[0].score(sk_le[1].transform(pd.DataFrame(LOAN)[LOAN_FEATURES]), pd.DataFrame(LOAN)[LOAN_TARGET]):.2f}   "
      f"sklearn gini {sk_lg[0].score(sk_lg[1].transform(pd.DataFrame(LOAN)[LOAN_FEATURES]), pd.DataFrame(LOAN)[LOAN_TARGET]):.2f}")

rows = []
for s in NEW_LOAN:
    rows.append({
        **s,
        "C4.5": predict_c45(loan_c45, s),
        "CART": predict_cart(loan_cart, s),
        "sklearn entropy": sklearn_predict(sk_le, s),
        "sklearn gini": sklearn_predict(sk_lg, s),
    })
display(pd.DataFrame(rows))


,Income,Credit,Loan
0,Low,Good,Yes
1,Low,Bad,No
2,Medium,Good,Yes
3,Medium,Bad,Yes
4,High,Good,Yes
5,High,Bad,No


From-scratch C4.5:
[Credit]  IG=0.4591  gainRatio=0.4591  n=6
  -> Credit = Good
    Leaf -> Yes   n=3  {'Yes': 3}
  -> Credit = Bad
    [Income]  IG=0.9183  gainRatio=0.5794  n=3
      -> Income = Low
        Leaf -> No   n=1  {'No': 1}
      -> Income = Medium
        Leaf -> Yes   n=1  {'Yes': 1}
      -> Income = High
        Leaf -> No   n=1  {'No': 1}

From-scratch CART:
[Credit]  weightedGini=0.2222  n=6
  -> Credit in ['Bad']
    [Income]  weightedGini=0.0  n=3
      -> Income in ['High', 'Low']
        Leaf -> No   n=2  {'No': 2}
      -> Income not in ['High', 'Low']
        Leaf -> Yes   n=1  {'Yes': 1}
  -> Credit not in ['Bad']
    Leaf -> Yes   n=3  {'Yes': 3}

sklearn entropy:
|--- Credit_Good <= 0.50
|   |--- Income_Medium <= 0.50
|   |   |--- class: No
|   |--- Income_Medium >  0.50
|   |   |--- class: Yes
|--- Credit_Good >  0.50
|   |--- class: Yes

sklearn gini:
|--- Credit_Good <= 0.50
|   |--- Income_Medium <= 0.50
|   |   |--- class: No
|   |--- Income_Medium >  

/usr/lib/python3.14/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/usr/lib/python3.14/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


,Income,Credit,C4.5,CART,sklearn entropy,sklearn gini
0,Medium,Bad,Yes,Yes,Yes,Yes
1,High,Bad,No,No,No,No
2,Low,Good,Yes,Yes,Yes,Yes


## 7. What the results say

**Play Tennis, C4.5.** Outlook wins the root (IG 0.2467, the only attribute above the average-gain filter). Overcast is pure Yes. Sunny then splits on Humidity at 77.5. Rain splits on Wind. Training accuracy is 14/14. A new Sunny day with humidity 80 is No; a Rain day with strong wind is No; Overcast is Yes.

**Play Tennis, CART.** The root is the same binary question (Overcast vs not). Below that, unrestricted Gini keeps cutting Temperature until every leaf is pure, so the tree memorizes temperature bands and never tests Humidity or Wind. Training accuracy is also 14/14, but Sunny/70/80/Weak and Rain/68/75/Strong are classified Yes, disagreeing with C4.5.

**sklearn.** Both criteria fit the 14 rows. Because sklearn only makes binary splits and the entropy criterion is information gain rather than gain ratio, its tree is closer to the from-scratch CART tree than to C4.5.

**Loan.** All four models recover the same rule and agree on the new samples: Credit = Good → Yes; Credit = Bad and Income = Medium → Yes; Credit = Bad and Income in {Low, High} → No.
